# Notebook 1: Data Exploration

**Goal:** Load the Food.com dataset, understand its structure, and prepare a clean working subset for experiments.

**Dataset:** Food.com Recipes and Interactions (Kaggle) — 230K recipes, 700K user interactions with ratings.

After this notebook you will have:
- `data/processed/recipes.parquet` — cleaned recipes with parsed nutrition
- `data/processed/interactions_train/val/test.parquet` — temporal train/val/test split

In [ ]:
# ============================================================
# SETUP: clone repo, install deps, add src to path
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

# Auto-detect environment
ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content'
REPO_DIR = f'{BASE}/{REPO}'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow matplotlib seaborn umap-learn plotly scikit-learn tqdm kaggle')
sys.path.insert(0, f'{REPO_DIR}/src')
print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")

In [ ]:
# ============================================================
# DATA PATHS - auto-detected for Colab and Kaggle
# ============================================================
import os

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'

if ENV == 'kaggle':
    # Kaggle: save processed data to working directory
    PROCESSED_DIR = '/kaggle/working/processed'
    # Food.com dataset added via "Add Data" in Kaggle
    KAGGLE_INPUT = '/kaggle/input/food-com-recipes-and-user-interactions'
    RAW_RECIPES = f'{KAGGLE_INPUT}/RAW_recipes.csv'
    RAW_INTERACTIONS = f'{KAGGLE_INPUT}/RAW_interactions.csv'
    print(f"Kaggle input: {KAGGLE_INPUT}")
else:
    # Colab: mount Google Drive to persist data between sessions
    from google.colab import drive
    drive.mount('/content/drive')
    PROCESSED_DIR = '/content/drive/MyDrive/food-recsys/processed'
    RAW_RECIPES = 'data/RAW_recipes.csv'
    RAW_INTERACTIONS = 'data/RAW_interactions.csv'

os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Processed data: {PROCESSED_DIR}")

## 1. Download Food.com Dataset

Option A (Kaggle API — recommended):

In [ ]:
# Download Food.com dataset
# - Kaggle: dataset added via "Add Data" button (no download needed)
# - Colab: download via Kaggle API (requires kaggle.json)
import os

if ENV == 'kaggle':
    print(f"Using Kaggle input dataset: {RAW_RECIPES}")
    assert os.path.exists(RAW_RECIPES), f"Add Food.com dataset via 'Add Data' button in Kaggle"
else:
    # Colab: requires kaggle.json uploaded to Files panel
    os.makedirs('data', exist_ok=True)
    os.system('kaggle datasets download -d shuyangli94/food-com-recipes-and-user-interactions -p data/')
    os.system('unzip -qo data/food-com-recipes-and-user-interactions.zip -d data/')
    print("Download complete")

## 2. Load and Inspect

In [ ]:
import pandas as pd
import ast

recipes_raw = pd.read_csv('data/RAW_recipes.csv')
interactions_raw = pd.read_csv('data/RAW_interactions.csv')

print(f"Recipes:      {recipes_raw.shape}")
print(f"Interactions: {interactions_raw.shape}")
print("\nRecipes columns:", list(recipes_raw.columns))
print("\nInteractions columns:", list(interactions_raw.columns))
recipes_raw.head(3)

## 3. Basic EDA

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
interactions_raw['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Reviews per recipe (log scale)
reviews_per_recipe = interactions_raw.groupby('recipe_id').size()
axes[1].hist(reviews_per_recipe, bins=50, log=True, color='steelblue', edgecolor='white')
axes[1].set_title('Reviews per Recipe (log scale)')
axes[1].set_xlabel('Number of reviews')
axes[1].set_ylabel('Count (log)')

# Reviews per user (log scale)
reviews_per_user = interactions_raw.groupby('user_id').size()
axes[2].hist(reviews_per_user, bins=50, log=True, color='coral', edgecolor='white')
axes[2].set_title('Reviews per User (log scale)')
axes[2].set_xlabel('Number of reviews')
axes[2].set_ylabel('Count (log)')

plt.tight_layout()
plt.show()

print(f"\nRating distribution:")
print(interactions_raw['rating'].value_counts(normalize=True).sort_index().map('{:.1%}'.format))

## 4. Parse Nutrition

The `nutrition` column is stored as a Python list string: `[calories, total_fat, sugar, sodium, protein, saturated_fat, carbohydrates]`.

In [ ]:
def parse_nutrition(nutrition_str):
    """Parse nutrition list string into a dict."""
    try:
        vals = ast.literal_eval(nutrition_str)
        return {
            'calories': float(vals[0]),
            'fat_g': float(vals[1]),
            'sugar_g': float(vals[2]),
            'sodium_mg': float(vals[3]),
            'protein_g': float(vals[4]),
            'sat_fat_g': float(vals[5]),
            'carbs_g': float(vals[6]),
        }
    except Exception:
        return {}

nutrition_df = pd.DataFrame(recipes_raw['nutrition'].apply(parse_nutrition).tolist())
print("Nutrition stats:")
print(nutrition_df.describe().round(1))

# Correlation matrix
import seaborn as sns
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(nutrition_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Nutrition Feature Correlations')
plt.tight_layout()
plt.show()

## 5. Tags Analysis

In [ ]:
from collections import Counter

all_tags = []
for tags_str in recipes_raw['tags'].dropna():
    try:
        all_tags.extend(ast.literal_eval(tags_str))
    except Exception:
        pass

tag_counts = Counter(all_tags)
top_tags = tag_counts.most_common(30)

fig, ax = plt.subplots(figsize=(12, 5))
names, counts = zip(*top_tags)
ax.bar(names, counts, color='steelblue', edgecolor='white')
ax.set_title('Top 30 Tags')
ax.set_ylabel('Recipe count')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

# Meal-time tags
meal_tags = ['breakfast', 'lunch', 'dinner', 'brunch', 'snacks']
print("\nMeal-time tag counts:")
for tag in meal_tags:
    print(f"  {tag}: {tag_counts.get(tag, 0):,}")

## 6. Dataset Limitations

> **This section is important for academic integrity — we discuss what the dataset CAN and CANNOT tell us.**

### Limitation 1: Rating ≠ Order (proxy metric gap)
Food.com contains recipe *ratings* (1–5 stars), not actual cooking or ordering events. A 5-star rating may indicate visual appeal or aspiration rather than actual preference. In production, the primary signal is order completion — a much stronger indicator. We treat ratings ≥ 4 as positive implicit feedback, acknowledging this approximation.

### Limitation 2: No User Profiles
Food.com provides no demographic data, health goals, dietary restrictions, allergies, or nutritional targets. Our production system relies heavily on user profiles. We evaluate embedding quality through **item-to-item co-preference**: do recipes liked by the same user cluster together?

### Limitation 3: Survivorship Bias
Users only rate recipes they chose to interact with. We never observe negative choices. Our negatives are *unrated* recipes — approximations, not true rejections.

### Limitation 4: Missing Temporal Context
No time-of-day, no what-was-eaten-before, no nutritional state. These are critical signals in production but cannot be evaluated here.

### Why Food.com is still useful
Despite limitations, Food.com is the largest open dataset with recipe texts AND user interaction data. It allows us to measure whether architectural changes to text representations improve retrieval quality. **The relative metric improvements (delta between configurations) are meaningful even if absolute metrics don't transfer to production.**

## 7. Prepare Working Subset

In [ ]:
import numpy as np
np.random.seed(42)

# Filter: users with ≥5 ratings, recipes with ≥3 ratings
user_counts = interactions_raw.groupby('user_id').size()
recipe_counts = interactions_raw.groupby('recipe_id').size()

active_users = user_counts[user_counts >= 5].index
popular_recipes = recipe_counts[recipe_counts >= 3].index

interactions_filtered = interactions_raw[
    interactions_raw['user_id'].isin(active_users) &
    interactions_raw['recipe_id'].isin(popular_recipes)
].copy()

print(f"Filtered interactions: {len(interactions_filtered):,} (from {len(interactions_raw):,})")
print(f"Unique users: {interactions_filtered['user_id'].nunique():,}")
print(f"Unique recipes: {interactions_filtered['recipe_id'].nunique():,}")

# Temporal split (60/20/20)
interactions_filtered['date'] = pd.to_datetime(interactions_filtered['date'])
interactions_filtered = interactions_filtered.sort_values('date')
n = len(interactions_filtered)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train = interactions_filtered.iloc[:train_end]
val   = interactions_filtered.iloc[train_end:val_end]
test  = interactions_filtered.iloc[val_end:]

print(f"\nSplit: train={len(train):,} | val={len(val):,} | test={len(test):,}")

# Build recipes dataframe with parsed nutrition and tags
recipe_ids_used = set(interactions_filtered['recipe_id'].unique())
recipes = recipes_raw[recipes_raw['id'].isin(recipe_ids_used)].copy()

nutr = pd.DataFrame(recipes['nutrition'].apply(parse_nutrition).tolist())
recipes = pd.concat([recipes.reset_index(drop=True), nutr], axis=1)

recipes['tag_list'] = recipes['tags'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

# Save
os.makedirs('data/processed', exist_ok=True)
recipes.to_parquet(f'{PROCESSED_DIR}/recipes.parquet', index=False)
train.to_parquet(f'{PROCESSED_DIR}/interactions_train.parquet', index=False)
val.to_parquet(f'{PROCESSED_DIR}/interactions_val.parquet', index=False)
test.to_parquet(f'{PROCESSED_DIR}/interactions_test.parquet', index=False)

print(f"\n✓ Saved processed data to data/processed/")
print(f"  recipes.parquet: {len(recipes):,} rows")